In [1]:
!pip install -q -U transformers accelerate

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import re
import json
import glob
import time
import subprocess
import numpy as np
import pandas as pd
import torch
from PIL import Image
from scipy.stats import spearmanr
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
device: cuda


In [2]:
MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"

processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=256 * 28 * 28,
    max_pixels=1024 * 28 * 28,
)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    max_memory={0: "12GiB", 1: "12GiB"},
    attn_implementation="sdpa",
)
model.eval()
for p_ in model.parameters():
    p_.requires_grad_(False)

lm = model.model.language_model
lm_head = model.lm_head
layers = lm.layers
n_layer = model.config.text_config.num_hidden_layers
n_head = model.config.text_config.num_attention_heads
hidden_dim = model.config.text_config.hidden_size
head_dim = layers[0].self_attn.o_proj.in_features // n_head
IMAGE_TOKEN_ID = model.config.image_token_id
assert len(layers) == n_layer
print(n_layer, "language model layers,", n_head, "heads,", head_dim, "head dim, image token id", IMAGE_TOKEN_ID, "| attention", model.config.text_config._attn_implementation)

preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

28 language model layers, 28 heads, 128 head dim, image token id 151655 | attention sdpa


In [3]:
INPUT = "/kaggle/input/datasets/nocturnalnerd18"
USERNAME = "rohanrbharadwaj"
AMBER_IMAGE_DIR = "/kaggle/input/datasets/nocturnalnerd18/amber-hallucination/image"

STAGE = "B"
SHARD = 2
N_SHARDS = 3
N_A_UNITS = 150
K_B = 40
N_B_UNITS = 1000
N_GATE = 6
A_SLUG = "nocturnalnerd18/patch-qwen-exact"
SLUG = A_SLUG if STAGE == "A" else f"{USERNAME}/patch-qwen-exact-b{SHARD}"
assert 0 <= SHARD < N_SHARDS
MAX_HOURS = 9.0
ALLOW_MISSING_IMAGES = True
PILOT_UNITS = 3
BATCH_CELLS = 8
BATCH_SAVE_SIZE = 50
UPLOAD_EVERY = 2
CHECK_TOL = 0.25
TAIL_N = 5
GROUPS = ["visual", "text", "tail"]
OUT_DIR = "/kaggle/working/patch_qwen"
os.makedirs(OUT_DIR, exist_ok=True)
CHECK_CELL = {"kind": "rp", "layer": 0, "head": -1, "group": "visual"}


def load_image(path):
    return Image.open(path).convert("RGB")


def build_inputs(image, text):
    conversation = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": text}]}]
    prompt_text = processor.apply_chat_template(conversation, add_generation_prompt=True)
    return processor(text=prompt_text, images=image, return_tensors="pt").to(model.device, torch.float16)


VG_DATASET = "sajjad2goudarzi/the-visual-genome-dataset"
VG_LOCAL = {}
vg_roots = {d_ for d_ in (f"/kaggle/input/datasets/{VG_DATASET}", f"/kaggle/input/{VG_DATASET.split(chr(47))[1]}") if os.path.isdir(d_)}
for pat in ["/kaggle/input/*", "/kaggle/input/*/*", "/kaggle/input/datasets/*/*"]:
    for d_ in glob.glob(pat):
        name = os.path.basename(d_).lower()
        if os.path.isdir(d_) and ("genome" in name or name.startswith("vg") or name.startswith("visualgenome")):
            vg_roots.add(d_)
for pat in ["/kaggle/input/*/VG_100K*", "/kaggle/input/*/*/VG_100K*", "/kaggle/input/*/*/*/VG_100K*", "/kaggle/input/datasets/*/*/VG_100K*", "/kaggle/input/datasets/*/*/*/VG_100K*"]:
    vg_roots.update(d_ for d_ in glob.glob(pat) if os.path.isdir(d_))
for root_ in sorted(vg_roots):
    for dp, dn, fns in os.walk(root_):
        for fn in fns:
            if fn.lower().endswith(".jpg") and fn[:-4].isdigit():
                VG_LOCAL[fn[:-4]] = os.path.join(dp, fn)
print(len(VG_LOCAL), "Visual Genome images found in attached datasets, roots:", sorted(vg_roots))
if not VG_LOCAL:
    for root_ in sorted(vg_roots) or ["/kaggle/input"]:
        for dp, dn, fns in os.walk(root_):
            if dp[len(root_):].count(os.sep) >= 2:
                dn[:] = []
            print("  ", dp, f"{len(dn)} dirs, {len(fns)} files", fns[:3])


def find_vg_image_path(image_id):
    return VG_LOCAL.get(str(image_id))


def resolve_image_path(q):
    qid = q["question_id"]
    if qid.startswith("amber_"):
        return os.path.join(AMBER_IMAGE_DIR, f"{q['image_id']}.jpg")
    if qid.startswith("reefknot_"):
        return find_vg_image_path(q["image_id"])
    return None


with open(f"{INPUT}/vlm-questions/all_questions.json") as f:
    all_questions = json.load(f)
assert len(all_questions) == 17492, "attach vlm-questions version 2"
question_by_id = {q["question_id"]: q for q in all_questions}
with open(f"{INPUT}/dla-qwen/dla_meta.json") as f:
    dla_meta = json.load(f)
PAIRS = {k: tuple(v) for k, v in dla_meta["pairs"].items()}
for prefix, (yes_id, no_id) in PAIRS.items():
    words = [processor.tokenizer.decode([i]).strip() for i in (yes_id, no_id)]
    print(prefix, yes_id, no_id, words)
    assert [w.lower() for w in words] == ["yes", "no"], words
assert set(PAIRS) == {"amber_", "reefknot_"}, set(PAIRS)


def batch_number(path):
    return int(re.search(r"_batch_(\d+)\.pt$", path).group(1))


atp = {}
atp_files = sorted(glob.glob(f"{INPUT}/atp-qwen-noising/**/*_batch_*.pt", recursive=True), key=lambda p: (os.path.basename(p).split("_batch_")[0], batch_number(p)))
n_failed = 0
for path in atp_files:
    for e in torch.load(path):
        if e["ok"]:
            atp[e["question_id"]] = {k: e[k] for k in (
                "question_id", "image_id", "hallucination_type", "stratum", "group", "source", "split", "sign", "seed",
                "delta_clean", "delta_corrupt", "TE", "noise_used", "head", "mlp",
            )}
        else:
            n_failed += 1
print(f"atp-qwen-noising: {len(atp_files)} batch files, {len(atp)} ok entries, {n_failed} failed entries")
if not atp_files:
    print("datasets visible under", INPUT, ":", sorted(os.listdir(INPUT)))
assert len(atp) >= 0.98 * 2927, f"only {len(atp)} ok atp entries from {len(atp_files)} files; attach atp-qwen-noising (all batches) under {INPUT}"

meta = pd.DataFrame([{k: v for k, v in a.items() if k not in ("head", "mlp")} for a in atp.values()])
meta["gap_clean"] = meta.sign * meta.delta_clean
fy_all = meta[meta.stratum == "false_yes"]
ty_all = meta[meta.stratum == "true_yes"]
rng = np.random.default_rng(0)
units = []
for (t, s, sp), g in fy_all.groupby(["hallucination_type", "source", "split"]):
    pool = ty_all[(ty_all.hallucination_type == t) & (ty_all.source == s) & (ty_all.split == sp)].set_index("question_id").gap_clean
    ids = g.question_id.tolist()
    gaps = g.set_index("question_id").gap_clean
    order = rng.permutation(len(ids))
    for rank, i in enumerate(order):
        qid = ids[i]
        ctrl = None
        if len(pool):
            ctrl = (pool - gaps[qid]).abs().idxmin()
            pool = pool.drop(ctrl)
        units.append({"fy": qid, "ctrl": ctrl, "key": (rank + 0.5) / len(ids), "stratum": f"{t}/{s}/{sp}"})
units.sort(key=lambda u: (u["key"], u["stratum"]))
print(len(units), "false_yes units,", sum(u["ctrl"] is not None for u in units), "with a matched true_yes control")
print(pd.crosstab(fy_all.hallucination_type, [fy_all.source, fy_all.split]))

stage_units = units[:N_A_UNITS] if STAGE == "A" else units[N_A_UNITS:N_A_UNITS + N_B_UNITS][SHARD::N_SHARDS]
stage_ids = [q for u in stage_units for q in (u["fy"], u["ctrl"]) if q is not None]
print(f"stage {STAGE}: {len(stage_units)} units, {len(stage_ids)} questions")
print(pd.Series([u["stratum"] for u in stage_units]).value_counts().to_string())

reef_ids = sorted({question_by_id[q]["image_id"] for q in stage_ids if q.startswith("reefknot_")})
n_ready = sum(find_vg_image_path(i) is not None for i in reef_ids)
print(f"{n_ready} of {len(reef_ids)} Visual Genome images found locally")
if n_ready < len(reef_ids):
    print("reefknot questions without an image are skipped now; attach the Visual Genome dataset (sajjad2goudarzi/the-visual-genome-dataset) to run them")
assert ALLOW_MISSING_IMAGES or n_ready >= 0.9 * len(reef_ids), "Visual Genome images are missing: attach the Visual Genome dataset or set ALLOW_MISSING_IMAGES = True"


def image_ok(qid):
    path = resolve_image_path(question_by_id[qid])
    return path is not None and os.path.exists(path)

with open(f"{INPUT}/atp-qwen-effects/shortlist.json") as f:
    shortlist = json.load(f)


def cell_of(index, group):
    if index < n_layer * n_head:
        return {"kind": "head", "layer": index // n_head, "head": index % n_head, "group": group}
    return {"kind": "mlp", "layer": index - n_layer * n_head, "head": -1, "group": group}


def cell_name(c):
    base = f"L{c['layer']} H{c['head']}" if c["kind"] == "head" else f"L{c['layer']} MLP"
    return f"{base} {c['group']}"


def build_full_cells():
    out = []
    for r in shortlist["ranked"]:
        groups = {"all"} | {k.split("/")[-1] for k in r["evidence"] if k.split("/")[-1] in GROUPS}
        for g in sorted(groups):
            out.append(cell_of(r["index"], g))
    for r in shortlist["always_include"]:
        out.append(cell_of(r["index"], r["group"]))
    seen, uniq = set(), []
    for c in out:
        key = cell_name(c)
        if key not in seen:
            seen.add(key)
            uniq.append(c)
    return uniq


if STAGE == "A":
    cells_run = build_full_cells()
else:
    cells_path = f"{INPUT}/patch-qwen-exact/cells_B.json"
    if not os.path.exists(cells_path):
        os.makedirs("/kaggle/temp/a_dl", exist_ok=True)
        subprocess.run(["kaggle", "datasets", "download", "-d", A_SLUG, "-p", "/kaggle/temp/a_dl", "--unzip"], capture_output=True, text=True)
        cells_path = "/kaggle/temp/a_dl/cells_B.json"
    assert os.path.exists(cells_path), "attach patch-qwen-exact (stage A, with cells_B.json) as an input"
    with open(cells_path) as f:
        cells_run = json.load(f)
    have = {cell_name(c) for c in cells_run}
    for r in shortlist["ranked"][:N_GATE]:
        c = cell_of(r["index"], "all")
        if cell_name(c) not in have:
            cells_run.append(c)
            have.add(cell_name(c))
print(len(cells_run), "patch cells per direction:", {k: sum(c["kind"] == k for c in cells_run) for k in ("head", "mlp")},
      "| groups", {g: sum(c["group"] == g for c in cells_run) for g in ["all"] + GROUPS})


def prep(qid):
    q = question_by_id[qid]
    inputs = build_inputs(load_image(resolve_image_path(q)), q["question_text"])
    ids = inputs["input_ids"]
    T = ids.shape[1]
    vis = ids[0] == IMAGE_TOKEN_ID
    tail = torch.zeros(T, dtype=torch.bool, device=ids.device)
    tail[-TAIL_N:] = True
    text = ~vis & ~tail
    assert int(vis.sum()) == int(inputs["image_grid_thw"].prod()) // 4 and not bool((vis & tail).any())
    yes_id, no_id = PAIRS[qid.split("_")[0] + "_"]
    return {
        "qid": qid, "T": T, "inputs": inputs, "vis": vis, "yes_id": yes_id, "no_id": no_id,
        "idx": {
            "all": torch.arange(T), "visual": torch.nonzero(vis.cpu())[:, 0],
            "text": torch.nonzero(text.cpu())[:, 0], "tail": torch.nonzero(tail.cpu())[:, 0],
        },
        "sign": 1.0 if q["ground_truth_answer"].strip().lower() == "yes" else -1.0,
    }


108249 Visual Genome images found in attached datasets, roots: ['/kaggle/input/datasets/sajjad2goudarzi/the-visual-genome-dataset', '/kaggle/input/datasets/sajjad2goudarzi/the-visual-genome-dataset/images/VG_100K', '/kaggle/input/datasets/sajjad2goudarzi/the-visual-genome-dataset/images2/VG_100K_2']
amber_ 9454 2753 ['Yes', 'No']
reefknot_ 9693 2152 ['yes', 'no']
atp-qwen-noising: 15 batch files, 2927 ok entries, 0 failed entries
1119 false_yes units, 889 with a matched true_yes control
source             amber           reefknot           
split               test train val     test train  val
hallucination_type                                    
attribute             11    89  13        0     0    0
relation               0     2   2      152   697  153
stage B: 323 units, 570 questions
relation/reefknot/train    194
relation/reefknot/test      52
relation/reefknot/val       42
attribute/amber/train       25
attribute/amber/test         4
attribute/amber/val          4
relation/ambe

In [4]:
STATE = {"noise": None, "seed": 0, "capture": False, "live": None, "vis": None, "patch": None, "kw": None}


def _layer_pre(l):
    def hook(module, args, kwargs):
        h = args[0] if args else kwargs["hidden_states"]
        changed = False
        if l == 0 and STATE["noise"] is not None:
            vis = STATE["vis"]
            h = h.clone()
            x = h[:, vis].float()
            generator = torch.Generator().manual_seed(STATE["seed"])
            noise = torch.randn(x.shape, generator=generator).to(h.device)
            h[:, vis] = (x + noise * x.std() * STATE["noise"]).to(h.dtype)
            changed = True
        if STATE["capture"]:
            STATE["live"]["rp"][l] = h
            if l == 0:
                STATE["kw"] = (tuple(args[1:]), {k: v for k, v in kwargs.items() if k != "hidden_states"})
        if not changed:
            return None
        if args:
            return (h,) + tuple(args[1:]), kwargs
        kwargs = dict(kwargs)
        kwargs["hidden_states"] = h
        return args, kwargs
    return hook


def _z_pre(l):
    def hook(module, args):
        z = args[0]
        patch = STATE["patch"]
        if patch is not None and patch["layer"] == l and patch["heads"]:
            z = z.clone()
            B, T = z.shape[0], z.shape[1]
            z4 = z.view(B, T, n_head, head_dim)
            src = patch["src"]["z"][l][0].view(T, n_head, head_dim)
            for b, h_, idx in patch["heads"]:
                ii = idx.to(z.device)
                z4[b, ii, h_] = src[ii.to(src.device), h_].to(z.device, z.dtype)
            return (z,) + tuple(args[1:])
        if STATE["capture"]:
            STATE["live"]["z"][l] = z
        return None
    return hook


def _mlp_fwd(l):
    def hook(module, args, output):
        patch = STATE["patch"]
        if patch is not None and patch["layer"] == l and patch["mlps"]:
            out = output.clone()
            src = patch["src"]["mlp"][l][0]
            for b, idx in patch["mlps"]:
                ii = idx.to(out.device)
                out[b, ii] = src[ii.to(src.device)].to(out.device, out.dtype)
            return out
        if STATE["capture"]:
            STATE["live"]["mlp"][l] = output
        return None
    return hook


_handles = []
for l, layer in enumerate(layers):
    _handles.append(layer.register_forward_pre_hook(_layer_pre(l), with_kwargs=True))
    _handles.append(layer.self_attn.o_proj.register_forward_pre_hook(_z_pre(l)))
    _handles.append(layer.mlp.register_forward_hook(_mlp_fwd(l)))
print(len(_handles), "hooks registered")

84 hooks registered


In [5]:
@torch.no_grad()
def capture_forward(p, noise, seed):
    STATE.update(noise=noise, seed=seed, capture=True, vis=p["vis"], patch=None)
    STATE["live"] = {"rp": [None] * n_layer, "z": [None] * n_layer, "mlp": [None] * n_layer}
    out = model(**p["inputs"], use_cache=False, logits_to_keep=1)
    logits = out.logits[0, -1].float()
    delta = p["sign"] * (logits[p["yes_id"]] - logits[p["no_id"]])
    result = {"acts": STATE["live"], "delta": float(delta), "kw": STATE["kw"]}
    STATE.update(capture=False, live=None, noise=None)
    del out, logits
    return result


@torch.no_grad()
def patched_deltas(p, base, src, chunk):
    B = len(chunk)
    l0 = chunk[0]["layer"]
    h = base["acts"]["rp"][l0].expand(B, -1, -1).clone()
    heads, mlps = [], []
    for b, c in enumerate(chunk):
        assert c["layer"] == l0
        idx = p["idx"][c["group"]]
        if c["kind"] == "head":
            heads.append((b, c["head"], idx))
        elif c["kind"] == "mlp":
            mlps.append((b, idx))
        else:
            ii = idx.to(h.device)
            h[b, ii] = src["acts"]["rp"][l0][0, ii.to(src["acts"]["rp"][l0].device)].to(h.device, h.dtype)
    STATE.update(capture=False, noise=None, patch={"layer": l0, "heads": heads, "mlps": mlps, "src": src["acts"]})
    a_rest, kw = base["kw"]
    for l in range(l0, n_layer):
        out = layers[l](h, *a_rest, **kw)
        h = out[0] if isinstance(out, tuple) else out
    STATE["patch"] = None
    logits = lm_head(lm.norm(h[:, -1])).float()
    return (p["sign"] * (logits[:, p["yes_id"]] - logits[:, p["no_id"]])).cpu().numpy()


def run_cells(p, base, src, cell_list):
    vals = np.zeros(len(cell_list))
    by_layer = {}
    for i, c in enumerate(cell_list):
        by_layer.setdefault(c["layer"], []).append(i)
    for l in sorted(by_layer):
        idxs = by_layer[l]
        for k in range(0, len(idxs), BATCH_CELLS):
            part = idxs[k:k + BATCH_CELLS]
            vals[part] = patched_deltas(p, base, src, [cell_list[i] for i in part])
    return vals


def exact_question(p, seed, noise, cell_list):
    clean = capture_forward(p, None, seed)
    corrupt = capture_forward(p, noise, seed)
    run = cell_list + [CHECK_CELL]
    v_noise = run_cells(p, clean, corrupt, run)
    v_denoise = run_cells(p, corrupt, clean, run)
    result = {
        "delta_clean": clean["delta"], "delta_corrupt": corrupt["delta"],
        "noising": torch.from_numpy((v_noise[:-1] - clean["delta"]).astype(np.float32)),
        "denoising": torch.from_numpy((v_denoise[:-1] - corrupt["delta"]).astype(np.float32)),
        "check_noising": float(v_noise[-1] - corrupt["delta"]), "check_denoising": float(v_denoise[-1] - clean["delta"]),
    }
    del clean, corrupt
    return result


def exact_with_retry(p, seed, noise, cell_list):
    global BATCH_CELLS
    for attempt in range(2):
        try:
            return exact_question(p, seed, noise, cell_list), None
        except torch.cuda.OutOfMemoryError:
            STATE.update(capture=False, live=None, noise=None, patch=None)
            torch.cuda.empty_cache()
            BATCH_CELLS = max(2, BATCH_CELLS // 2)
    return None, "oom"


def atp_value(a, c):
    x = a["head"][:, c["layer"], c["head"]] if c["kind"] == "head" else a["mlp"][:, c["layer"]]
    return float(x.sum() if c["group"] == "all" else x[GROUPS.index(c["group"])])

In [6]:
pilot_ids = [q for u in stage_units for q in (u["fy"], u["ctrl"]) if q is not None and image_ok(q)][:2 * PILOT_UNITS]
rows, secs, checks, base_err = [], [], [], []
for qid in pilot_ids:
    a = atp[qid]
    p = prep(qid)
    t0 = time.time()
    res, reason = exact_with_retry(p, a["seed"], a["noise_used"], cells_run)
    assert res is not None, reason
    secs.append(time.time() - t0)
    checks.append(max(abs(res["check_noising"]), abs(res["check_denoising"])))
    base_err.append(max(abs(res["delta_clean"] - a["delta_clean"]), abs(res["delta_corrupt"] - a["delta_corrupt"])))
    for i, c in enumerate(cells_run):
        rows.append({"qid": qid, "cell": cell_name(c), "kind": c["kind"], "atp": atp_value(a, c),
                     "noising": float(res["noising"][i]), "denoising": float(res["denoising"][i])})
    print(f"{qid} {a['stratum']:9s} T={p['T']}: {secs[-1]:.1f} s | clean {res['delta_clean']:+.2f} (atp run {a['delta_clean']:+.2f}) "
          f"corrupt {res['delta_corrupt']:+.2f} ({a['delta_corrupt']:+.2f}) | identity check noising {res['check_noising']:+.3f} denoising {res['check_denoising']:+.3f}")
    del p, res
    torch.cuda.empty_cache()

pil = pd.DataFrame(rows)
print(f"clean and corrupt deltas vs the attribution patching run: max abs diff {max(base_err):.3f}")
print(f"identity checks (patching the whole noise back in or out must reproduce the other run): max abs error {max(checks):.3f}")
print(f"pilot speed: {np.mean(secs):.1f} s per question for {len(cells_run)} cells in both directions")
print(f"projected time for stage {STAGE}: {np.mean(secs) * len(stage_ids) / 3600:.1f} h for {len(stage_ids)} questions")
print(f"attribution patching vs exact noising over {len(pil)} cells: spearman {spearmanr(pil.atp, pil.noising)[0]:.3f}, "
      f"slope of exact on atp {(pil.atp * pil.noising).sum() / (pil.atp ** 2).sum():.3f}")
for kind, sub in pil.groupby("kind"):
    print(f"  {kind:5s} spearman {spearmanr(sub.atp, sub.noising)[0]:.3f} over {len(sub)} cells")
print(f"noising vs denoising effects: spearman {spearmanr(pil.noising, pil.denoising)[0]:.3f}")
big = pil.reindex(pil.noising.abs().sort_values(ascending=False).index).head(12)
print(big[["qid", "cell", "atp", "noising", "denoising"]].round(3).to_string(index=False))
assert max(checks) < CHECK_TOL, "the batched patched forward does not reproduce the full forward"
assert max(base_err) < CHECK_TOL, "clean or corrupt deltas differ from the attribution patching collection"

/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(


amber_1734 false_yes T=296: 11.4 s | clean -6.56 (atp run -6.56) corrupt -2.27 (-2.27) | identity check noising +0.000 denoising -0.016
amber_7091 true_yes  T=535: 18.0 s | clean +6.58 (atp run +6.58) corrupt +2.23 (+2.23) | identity check noising +0.000 denoising +0.000
reefknot_2406167_1 false_yes T=304: 10.0 s | clean -2.41 (atp run -2.41) corrupt -1.09 (-1.09) | identity check noising +0.016 denoising +0.000
reefknot_2395867_0 true_yes  T=304: 10.0 s | clean +2.41 (atp run +2.41) corrupt -1.11 (-1.11) | identity check noising -0.016 denoising +0.000
reefknot_2409580_1 false_yes T=316: 10.3 s | clean -1.16 (atp run -1.16) corrupt -0.95 (-0.95) | identity check noising +0.000 denoising +0.000
reefknot_2384292_0 true_yes  T=326: 10.7 s | clean +1.17 (atp run +1.17) corrupt +0.17 (+0.17) | identity check noising +0.000 denoising -0.016
clean and corrupt deltas vs the attribution patching run: max abs diff 0.000
identity checks (patching the whole noise back in or out must reproduce the

In [7]:
def _run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    cleaned = "\n".join(l for l in out.splitlines() if "upload info" not in l.lower()).lower()
    bad = ("client error", "forbidden", "unauthorized", "traceback", "error:", "failed")
    return r.returncode == 0 and not any(x in cleaned for x in bad), out


ok, out = _run(["kaggle", "datasets", "download", "-d", SLUG, "-p", OUT_DIR, "--unzip"])
if ok:
    print("restored", len(glob.glob(os.path.join(OUT_DIR, "*_batch_*.pt"))), "batch files from", SLUG)
else:
    if not any(s in out.lower() for s in ("404", "403", "not found", "does not exist", "forbidden")):
        raise RuntimeError("download failed for an unrelated reason, refusing to start fresh:\n" + out)
    print("no existing dataset, starting fresh")
dataset_exists = ok

with open(os.path.join(OUT_DIR, "dataset-metadata.json"), "w") as f:
    json.dump({
        "title": SLUG.split("/")[1],
        "id": SLUG,
        "subtitle": "Exact activation patching in both directions for Qwen2.5-VL yes/no questions",
        "description": "Exact activation patching on Qwen2.5-VL-1.5-7B for every false_yes question of atp-qwen-selection plus a true_yes control matched on the clean gap "
                       "within type, source and split. Corrupt run: Gaussian noise on the visual embeddings at layer 0 (noise_used times their std, seed from atp-qwen-noising). "
                       "noising [n_cells] is the change in delta when one component is replaced by its corrupt activation in the clean run (necessity), "
                       "denoising [n_cells] the change when it is replaced by its clean activation in the corrupt run (sufficiency). "
                       "delta is the sign-corrected Yes minus No logit gap at the last position. Cells are listed in cells_A.json and cells_B.json (kind head, mlp; layer; head; group all, visual, text, tail). "
                       "Stage A patches the full shortlist on the first units, stage B only the cells selected from stage A. check_noising and check_denoising are identity checks that must be near zero.",
        "licenses": [{"name": "CC0-1.0"}],
    }, f)


def upload(msg):
    global dataset_exists
    version_cmd = ["kaggle", "datasets", "version", "-p", OUT_DIR, "-m", msg]
    create_cmd = ["kaggle", "datasets", "create", "-p", OUT_DIR]
    ok_, out_ = _run(version_cmd if dataset_exists else create_cmd)
    if not ok_ and not dataset_exists:
        time.sleep(30)
        ok_, out2 = _run(version_cmd)
        out_ += "\n--- version retry ---\n" + out2
    if ok_:
        dataset_exists = True
        print(f"    [upload ok] {msg}")
    else:
        cleaned = [l for l in out_.splitlines() if l.strip() and "upload info" not in l.lower() and "B/s]" not in l and "it/s]" not in l]
        print(f"    [UPLOAD FAILED, continuing] account {USERNAME}, dataset {SLUG}\n" + "\n".join(cleaned[-8:]))
    return ok_


with open(os.path.join(OUT_DIR, f"cells_{STAGE}.json"), "w") as f:
    json.dump(cells_run, f)


def collect():
    paths = sorted(glob.glob(os.path.join(OUT_DIR, f"{STAGE}_batch_*.pt")), key=batch_number)
    done_ids = set()
    for path in paths:
        done_ids.update(e["question_id"] for e in torch.load(path))
    todo = [q for q in stage_ids if q not in done_ids]
    print(f"stage {STAGE}: {len(stage_ids)} questions, {len(done_ids)} already done, {len(todo)} to run")

    batch_entries, batch_index, failed, missing = [], len(paths), 0, 0
    t_start = time.time()
    stopped = False
    for i, qid in enumerate(todo):
        q = question_by_id[qid]
        image_path = resolve_image_path(q)
        if image_path is None or not os.path.exists(image_path):
            missing += 1
            continue
        a = atp[qid]
        p = prep(qid)
        res, reason = exact_with_retry(p, a["seed"], a["noise_used"], cells_run)
        entry = {
            "question_id": qid, "image_id": q["image_id"], "hallucination_type": q["hallucination_type"],
            "stratum": a["stratum"], "group": a["group"], "source": a["source"], "split": a["split"], "stage": STAGE,
            "sign": p["sign"], "seed": a["seed"], "noise_used": a["noise_used"], "T": p["T"], "ok": res is not None,
            "atp_delta_clean": a["delta_clean"], "atp_delta_corrupt": a["delta_corrupt"],
        }
        if res is None:
            entry["reason"] = reason
            failed += 1
        else:
            entry.update(res)
        batch_entries.append(entry)
        del p

        if len(batch_entries) >= BATCH_SAVE_SIZE:
            torch.save(batch_entries, os.path.join(OUT_DIR, f"{STAGE}_batch_{batch_index}.pt"))
            batch_entries, batch_index = [], batch_index + 1
            if batch_index % UPLOAD_EVERY == 0:
                upload(f"stage {STAGE}: {batch_index} batches")
        if i % 25 == 0 and i > 0:
            torch.cuda.empty_cache()
            avg = (time.time() - t_start) / (i + 1)
            print(f"[{i + 1}/{len(todo)}] {qid} ({avg:.1f} s/question avg, ETA {avg * (len(todo) - i - 1) / 3600:.1f} h, failed {failed}, missing images {missing})")
        if time.time() - t_start > MAX_HOURS * 3600:
            stopped = True
            print(f"time budget of {MAX_HOURS} h reached after {i + 1} questions; re-run this notebook to resume")
            break

    if batch_entries:
        torch.save(batch_entries, os.path.join(OUT_DIR, f"{STAGE}_batch_{batch_index}.pt"))
    upload(f"stage {STAGE}: {'paused' if stopped else 'done'}")
    print(f"done -- stage {STAGE}, failed {failed}, missing images {missing}, {(time.time() - t_start) / 3600:.2f} h, complete {not stopped}")
    return missing, stopped


missing, stopped = collect()
assert ALLOW_MISSING_IMAGES or missing <= 0.02 * len(stage_ids), f"{missing} questions have no image"
if missing:
    print(f"{missing} questions were skipped for lack of an image; re-run this notebook with the images reachable to finish them")

no existing dataset, starting fresh
stage B: 570 questions, 0 already done, 570 to run


/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(


[26/570] reefknot_2367588_0 (12.7 s/question avg, ETA 1.9 h, failed 0, missing images 0)
[51/570] reefknot_2400612_1 (12.5 s/question avg, ETA 1.8 h, failed 0, missing images 0)
[76/570] reefknot_2386553_0 (12.2 s/question avg, ETA 1.7 h, failed 0, missing images 0)
    [upload ok] stage B: 2 batches


/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(


[101/570] amber_1156 (12.5 s/question avg, ETA 1.6 h, failed 0, missing images 0)
[126/570] reefknot_2316437_0 (12.2 s/question avg, ETA 1.5 h, failed 0, missing images 0)
[151/570] reefknot_2387171_1 (12.1 s/question avg, ETA 1.4 h, failed 0, missing images 0)
[176/570] reefknot_2388750_0 (12.3 s/question avg, ETA 1.3 h, failed 0, missing images 0)
    [upload ok] stage B: 4 batches


...

[301/570] reefknot_2321782_1 (12.1 s/question avg, ETA 0.9 h, failed 0, missing images 0)
[326/570] reefknot_2365435_0 (12.1 s/question avg, ETA 0.8 h, failed 0, missing images 0)
[351/570] reefknot_2355235_1 (12.1 s/question avg, ETA 0.7 h, failed 0, missing images 0)
[376/570] reefknot_2329310_0 (12.0 s/question avg, ETA 0.6 h, failed 0, missing images 0)
    [upload ok] stage B: 8 batches


/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(


[401/570] reefknot_2404620_1 (12.0 s/question avg, ETA 0.6 h, failed 0, missing images 0)
[426/570] reefknot_2413035_0 (12.1 s/question avg, ETA 0.5 h, failed 0, missing images 0)
[451/570] reefknot_2320662_1 (12.1 s/question avg, ETA 0.4 h, failed 0, missing images 0)
[476/570] reefknot_2409784_1 (12.2 s/question avg, ETA 0.3 h, failed 0, missing images 0)
    [upload ok] stage B: 10 batches


/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(


[501/570] reefknot_2326284_1 (12.2 s/question avg, ETA 0.2 h, failed 0, missing images 0)
[526/570] reefknot_2385288_1 (12.2 s/question avg, ETA 0.1 h, failed 0, missing images 0)
[551/570] reefknot_2363220_1 (12.2 s/question avg, ETA 0.1 h, failed 0, missing images 0)
    [upload ok] stage B: done
done -- stage B, failed 0, missing images 0, 1.93 h, complete True


In [8]:
entries = []
for path in sorted(glob.glob(os.path.join(OUT_DIR, f"{STAGE}_batch_*.pt")), key=batch_number):
    entries.extend(torch.load(path))
ids = [e["question_id"] for e in entries]
print(f"{len(entries)} entries, {len(set(ids))} unique, {len(set(stage_ids) - set(ids))} stage questions not done yet")
assert len(ids) == len(set(ids)), "duplicated entries"
good = [e for e in entries if e["ok"]]
print(f"ok {len(good)}, failed {len(entries) - len(good)}")
print(pd.crosstab([pd.Series([e["hallucination_type"] for e in good], name="type"), pd.Series([e["source"] for e in good], name="source")],
                  pd.Series([e["stratum"] for e in good], name="stratum")))

check = max(max(abs(e["check_noising"]), abs(e["check_denoising"])) for e in good)
base = max(max(abs(e["delta_clean"] - e["atp_delta_clean"]), abs(e["delta_corrupt"] - e["atp_delta_corrupt"])) for e in good)
print(f"identity checks: max abs error {check:.3f} | clean and corrupt deltas vs the attribution patching run: max abs diff {base:.3f}")
bad_checks = [e["question_id"] for e in good if max(abs(e["check_noising"]), abs(e["check_denoising"])) > CHECK_TOL]
print(f"{len(bad_checks)} entries above the tolerance {CHECK_TOL}")
assert len(bad_checks) <= 0.02 * len(good), "too many entries fail the identity check"

N = np.stack([e["noising"].numpy() for e in good])
Dn = np.stack([e["denoising"].numpy() for e in good])
sg = np.array([e["sign"] for e in good])
A_pred = np.array([[atp_value(atp[e["question_id"]], c) for c in cells_run] for e in good])
stratum = np.array([e["stratum"] for e in good])
print(f"attribution patching vs exact noising over {N.size} entry-cells: spearman {spearmanr(A_pred.ravel(), N.ravel())[0]:.3f}, "
      f"slope {(A_pred * N).sum() / (A_pred ** 2).sum():.3f}")
mean_abs = np.abs(N).mean(0)
print(f"noising vs denoising, per-cell mean effect: spearman {spearmanr(N.mean(0), Dn.mean(0))[0]:.3f}")

if STAGE == "A":
    S_n = -sg[:, None] * N
    S_d = sg[:, None] * Dn
    fy_m, ct_m = stratum == "false_yes", stratum == "true_yes"
    assert fy_m.sum() >= 30 and ct_m.sum() >= 30, "too few stage A questions to choose stage B cells"

    def cohen(S):
        a, b = S[fy_m], S[ct_m]
        sp = np.sqrt(((len(a) - 1) * a.var(0, ddof=1) + (len(b) - 1) * b.var(0, ddof=1)) / (len(a) + len(b) - 2))
        return (a.mean(0) - b.mean(0)) / np.maximum(sp, 1e-6)

    d_score = np.maximum(np.abs(cohen(S_n)), np.abs(cohen(S_d)))
    mag_score = np.maximum(np.abs(S_n[fy_m].mean(0)), np.abs(S_d[fy_m].mean(0)))
    chosen = []
    for score in (d_score, mag_score):
        for i in np.argsort(-score):
            if int(i) not in chosen:
                chosen.append(int(i))
            if len(chosen) >= (K_B // 2 if score is d_score else K_B):
                break
    chosen = sorted(chosen)
    table = pd.DataFrame({
        "cell": [cell_name(cells_run[i]) for i in chosen], "d_score": d_score[chosen].round(2), "fy_mean_support_noising": S_n[fy_m].mean(0)[chosen].round(3),
        "fy_mean_support_denoising": S_d[fy_m].mean(0)[chosen].round(3),
    })
    print(table.sort_values("d_score", ascending=False).to_string(index=False))
    with open(os.path.join(OUT_DIR, "cells_B.json"), "w") as f:
        json.dump([cells_run[i] for i in chosen], f)
    print(len(chosen), "cells written to cells_B.json for stage B")

with open(os.path.join(OUT_DIR, f"patch_meta_{STAGE}.json"), "w") as f:
    json.dump({
        "model": MODEL_NAME, "pairs": PAIRS, "n_layer": n_layer, "n_head": n_head, "groups": GROUPS, "tail_n": TAIL_N,
        "stage": STAGE, "n_cells": len(cells_run), "n_ok": len(good), "n_failed": len(entries) - len(good), "max_check_error": check,
        "complete": len(set(stage_ids) - set(ids)) == 0,
    }, f)
if not upload(f"stage {STAGE}: verified"):
    print(f"final upload failed; the batch files and metadata are still in {OUT_DIR}, which is saved as this notebook's output")

570 entries, 570 unique, 0 stage questions not done yet
ok 570, failed 0
stratum             false_yes  true_yes
type      source                       
attribute amber            33        33
relation  amber             2         2
          reefknot        288       212
identity checks: max abs error 0.047 | clean and corrupt deltas vs the attribution patching run: max abs diff 0.000
0 entries above the tolerance 0.25
attribution patching vs exact noising over 23940 entry-cells: spearman 0.746, slope 0.457
noising vs denoising, per-cell mean effect: spearman -0.663
    [upload ok] stage B: verified
